In [1]:
import json
import re
import glob
import os
from collections import defaultdict
import unicodedata


In [2]:
def strip_accents(s: str) -> str:
    return "".join(
        c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c)
    )


def norm_tipo(s: str) -> str:
    return re.sub(r"\s+", " ", strip_accents(s or "")).strip().upper()


In [3]:
path_to_files = "/home/jovyan/data/processed/"

In [4]:
END_TYPE = re.compile(
    r"(?i)ADMINISTRANDO[\s,]+JUSTICIA[\s,]+EN[\s,]+NOMBRE[\s,]+DEL[\s,]+PUEBLO[\s,]+SOBERANO[\s,]+"
    r"DEL[\s,]+ECUADOR[\s,]+Y[\s,]+POR[\s,]+AUTORIDAD[\s,]+DE[\s,]+LA[\s,]+CONSTITUCI[Óó]N[\s,]+Y[\s,]+"
    r"LAS[\s,]+LEYES[\s,]+DE[\s,]+LA[\s,]+REP[Úú]BLICA[\s,]*(.*)", 
    re.DOTALL # Allows '.' to match newlines if the text continues on new lines
)

In [5]:
RE_PARTIAL = re.compile(
    r"ACEPT\w*\s+PARCIAL|PARCIALMENTE|ACCOGER?\s+PARCIAL"
)

# 2. Check this SECOND
RE_ACCEPT = re.compile(
    r"\b(ACEPT\w*|ACOG\w*)\s+(LA\s+|SU\s+)?DEMANDA"
    r"|\bSE\s+ACEPTA\b"
    r"|\bDECLARAR\s+(LA\s+)?VALIDEZ\b"  # Common alternative phrasing in EC courts
)

# 3. Check this THIRD
RE_DENY = re.compile(
    r"\b(NIEG\w*|NEG\w*|RECHAZ\w*|DESECH\w*)\s+(LA\s+|SU\s+)?DEMANDA"
    r"|\bSIN\s+LUGAR\b"
    r"|\bIMPROCEDENTE\b"
    r"|\bDECLARAR\s+SIN\s+LUGAR\b"
)


In [6]:
ruling_outcome = {"accept": 0, "partial": 0, "denied": 0, "no matches found": 0}

In [7]:
#find all jsonfiles in directory
json_files = glob.glob(os.path.join(path_to_files, "*.json"))

In [8]:
match_counter = 0
for file_path in json_files:
    with open(file_path, 'rb') as file:
        if file.read(2) != b'[]':
            file_name = os.path.basename(file_path)
            with open(file_path, "r", encoding="utf-8") as f:
                try:
                    data = json.load(f)
                    for item in data:
                        ruling = item.get("actividad")
                        relevant_ruling = ruling[-2500:]
                        clean_ruling = norm_tipo(relevant_ruling)
                        match = END_TYPE.search(ruling)
                        if match:
                            match_counter += 1
                            # .strip() cleans up any leading or trailing whitespaces/newlines
                            d = match.group(1).strip()
                            d = norm_tipo(d)
                            print(d[:300])
                            if RE_PARTIAL.search(d):
                                ruling_outcome["partial"] += 1
                                #print(f"partial {d}")
                            elif RE_ACCEPT.search(d):
                                ruling_outcome["accept"] += 1
                                #print(f"accept {d}")
                            elif RE_DENY.search(d):
                                ruling_outcome["denied"] += 1
                               #print(f"denied {d}")
                            else:
                                ruling_outcome["no matches found"] += 1
                        else:
                            print("No match found.")
                            #print(f"Successfully loaded: {file_name}")
                except json.JSONDecodeError:
                    print(f"Error: {file_name} is not a valid JSON file.")

RESUELVEN: ACEPTAR LA DEMANDA PRESENTADA POR LA SENORA NOEMI GUADALUPE FREIRE SALVADOR; EN CONSECUENCIA, SE DEJA SIN EFECTO LEGAL NI VALOR JURIDICO ALGUNO A LA RESOLUCION NUMERO SENAE-DDE-2019-0123-RE DE 2 DE JULIO DEL 2019, SUSCRITA POR LA DIRECTORA DISTRITAL DE ESMERALDAS DEL SERVICIO NACIONAL DE 
RESUELVE: 8.1. NEGAR LA DEMANDA PRESENTADA POR LA SRA. ZHUANG HUIMEI, POR SUS PROPIOS Y PERSONALES DERECHOS. 8.2. RATIFICAR LA VALIDEZ Y LEGALIDAD DE LA RECTIFICACION DE TRIBUTOS NO. JRP2-2018-0275-D001, EMITIDA POR LA DIRECTORA NACIONAL DE INTERVENCION DEL SENAE EL 27 DE JUNIO DE 2019. 8.3. DISPONE
ACEPTAMOS PARCIALMENTE LA DEMANDA PRESENTADA POR EL DR. ANGEL DIEGO ALMEIDA GUZMAN, PROCURADOR JUDICIAL DE, CONSORCIO LINEA 1-METRO DE QUITO ACCIONA, EN CONSECUENCIA, SE DECLARA LA ILEGITIMIDAD DE LAS GLOSAS: 3.6.4.4, DENOMINADA “PAGOS AL PERSONAL EXPATRIADO, SIN SUSTENTO DE LA PRESTACION DEL SERVIC


In [9]:
print(ruling_outcome)

{'accept': 1, 'partial': 1, 'denied': 1, 'no matches found': 0}


In [10]:
print(match_counter)

3


In [11]:

# if "partial" in hits:
#     return "partial"
# if len(hits) == 1:
#     return next(iter(hits))
# return "ambiguous" if hits else "unknown"